In [0]:
from pyspark.sql.functions import col, to_date, row_number
from pyspark.sql.window import Window

silver_schema = dbutils.widgets.get("silver_schema")
bronze_schema = dbutils.widgets.get("bronze_schema")

# Step 1: Ensure silver schema exists in default catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# Step 2: Load Bronze orders
bronze_orders = spark.table(f"{bronze_schema}.orders_bronze")

# Step 3: Rename and cast columns
orders_cleaned = (
    bronze_orders
    .select(
        col("OrderID").alias("order_id"),
        to_date(col("OrderDate")).alias("order_date"),
        col("CustomerID").alias("customer_id"),
        col("TotalAmount").cast("double").alias("total_amount"),
        col("Status").alias("status")
    )
)

# Step 4: Write to Silver layer
orders_cleaned.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.orders_cleaned")
print("✅ Silver cleaned complete: {silver_schema}.orders_cleaned")



✅ Silver cleaned complete: silver_schema.orders_cleaned


In [0]:
orders_cleaned.show(500)

+--------+----------+-----------+------------+---------+
|order_id|order_date|customer_id|total_amount|   status|
+--------+----------+-----------+------------+---------+
|       1|2023-01-01|         25|     6202.83|Cancelled|
|       2|2023-01-02|         32|     9871.87|Cancelled|
|       3|2023-01-03|         78|     6814.71|Completed|
|       4|2023-01-04|         89|     4972.72|  Pending|
|       5|2023-01-05|         91|      5150.9|  Pending|
|       6|2023-01-06|          6|     6603.76|  Pending|
|       7|2023-01-07|         20|     7080.93|Cancelled|
|       8|2023-01-08|         15|     4101.39|Cancelled|
|       9|2023-01-09|         16|     1755.19|  Pending|
|      10|2023-01-10|         34|     4953.16|  Pending|
|      11|2023-01-11|         49|     7449.77|Cancelled|
|      12|2023-01-12|         76|     9997.49|Cancelled|
|      13|2023-01-13|         38|     8166.68|Cancelled|
|      14|2023-01-14|         98|      605.24|Cancelled|
|      15|2023-01-15|         4

In [0]:
w = Window.partitionBy("order_id").orderBy(col("order_date").desc())

order_with_rn = orders_cleaned.withColumn("rn", row_number().over(w))

In [0]:
order_with_rn.show(500)

+--------+----------+-----------+------------+---------+---+
|order_id|order_date|customer_id|total_amount|   status| rn|
+--------+----------+-----------+------------+---------+---+
|       1|2023-01-01|         25|     6202.83|Cancelled|  1|
|       2|2023-01-02|         32|     9871.87|Cancelled|  1|
|       3|2023-01-03|         78|     6814.71|Completed|  1|
|       4|2023-01-04|         89|     4972.72|  Pending|  1|
|       5|2023-01-05|         91|      5150.9|  Pending|  1|
|       6|2023-01-06|          6|     6603.76|  Pending|  1|
|       7|2023-01-07|         20|     7080.93|Cancelled|  1|
|       8|2023-01-08|         15|     4101.39|Cancelled|  1|
|       9|2023-01-09|         16|     1755.19|  Pending|  1|
|      10|2023-01-10|         34|     4953.16|  Pending|  1|
|      11|2023-01-11|         49|     7449.77|Cancelled|  1|
|      12|2023-01-12|         76|     9997.49|Cancelled|  1|
|      13|2023-01-13|         38|     8166.68|Cancelled|  1|
|      14|2023-01-14|   

In [0]:
total_cleaned = orders_cleaned.count()
total_dedupe = order_with_rn.filter(col("rn") == 1).count()

duplicate_exists = total_cleaned != total_dedupe


In [0]:
dbutils.jobs.taskValues.set(key="has_duplicates", value=duplicate_exists)